# 08 — Chipotle Order Sorting & Multi-Condition Filtering
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Product Analytics, and Business Intelligence Interviews.*

---

## 📌 Executive Summary & Interview Expectations
This lab tests your ability to answer **ad-hoc business questions** on transactional data by combining **vectorized filtering**, **multi-stage sorting**, **deduplication**, and **safe currency parsing**.

### Core Competencies Tested in this Module:
1. **Safe Currency Normalization**: Idempotent parsing without brittle string index slicing.
2. **Unit Price Isolation**: Isolating base menu item prices by filtering for single-quantity purchases (`quantity == 1`) to eliminate multi-unit distortion.
3. **Menu Item Deduplication**: Using `drop_duplicates(subset=[...])` to extract distinct product/topping combinations.
4. **Order Count vs Quantity Sold**: Differentiating transaction occurrences (`len(df[mask])` or `mask.sum()`) from total volume (`df.loc[mask, 'quantity'].sum()`).
5. **Interview Corner**: `mask.sum()` vs `len(df[mask])` micro-optimizations, string filtering with `na=False`, and complex basket breakdowns.

## 1. Environment Setup & Data Ingestion

In [1]:
import os
import numpy as np
import pandas as pd

# Load Chipotle dataset (tab-delimited)
tsv_path = "chipotle.tsv"
if not os.path.exists(tsv_path):
    tsv_path = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"

chipo = pd.read_csv(tsv_path, sep="\t")
print(f"Data successfully loaded. Shape: {chipo.shape}")
chipo.head()

Data successfully loaded. Shape: (4622, 5)


,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98


## 2. Safe, Idempotent Currency Cleaning

### ⚠️ Top Interview Trap: Re-running String Transformations
- Hardcoding `apply(lambda x: float(x[1:]))` crashes on re-execution because the column is already a `float`.
- Vectorized `.str.replace('$', '', regex=False).str.strip().astype(float)` combined with `pd.api.types.is_numeric_dtype` is idempotent and runs 50x faster.

In [2]:
# Idempotent price cleaning
if not pd.api.types.is_numeric_dtype(chipo["item_price"]):
    chipo["item_price"] = (
        chipo["item_price"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.strip()
        .astype(float)
    )

print("item_price cleaned successfully. Dtype:", chipo["item_price"].dtype)

item_price cleaned successfully. Dtype: float64


## 3. Isolating Unit Prices & Premium Products (> $10.00)

### ⚠️ Top Interview Question: Why filter for `quantity == 1`?
In Chipotle order data, `item_price` is the total price for `quantity` units.
If a customer orders 2 sodas for $2.18, that row shows `item_price = 2.18`.
To find products that **cost more than $10.00 per unit**, we MUST restrict our baseline lookup to rows where `quantity == 1` and drop duplicate product variations!

In [3]:
# Filter for single-item purchases to isolate true base prices
single_items = chipo[chipo["quantity"] == 1].copy()

# Deduplicate to find unique menu items with their base price
unique_items = single_items.drop_duplicates(subset=["item_name"])

# Products costing more than $10.00
expensive_items = unique_items[unique_items["item_price"] > 10.0]
print(f"Number of distinct products costing > $10.00: {len(expensive_items)}")
display(expensive_items[["item_name", "item_price"]].sort_values(by="item_price", ascending=False))

Number of distinct products costing > $10.00: 12


,item_name,item_price
606,Steak Salad Bowl,11.89
1132,Carnitas Salad Bowl,11.89
1229,Barbacoa Salad Bowl,11.89
7,Steak Burrito,11.75
39,Barbacoa Bowl,11.75
168,Barbacoa Crispy Tacos,11.75
57,Veggie Burrito,11.25
62,Veggie Bowl,11.25
186,Veggie Salad Bowl,11.25
738,Veggie Soft Tacos,11.25


## 4. Customization Analysis: Varieties of 'Chicken Bowl'

In [4]:
# Distinct topping combinations for Chicken Bowls
chicken_bowl_varieties = (
    chipo[(chipo["item_name"] == "Chicken Bowl") & (chipo["quantity"] == 1)]
    .drop_duplicates(subset=["choice_description"])
)

print(f"Number of distinct Chicken Bowl topping combinations: {len(chicken_bowl_varieties)}")
chicken_bowl_varieties[["item_name", "choice_description", "item_price"]].head(5)

Number of distinct Chicken Bowl topping combinations: 333


,item_name,choice_description,item_price
5,Chicken Bowl,"[Fresh Tomato Salsa (Mild), [Rice, Cheese, Sou...",10.98
13,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",11.25
19,Chicken Bowl,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",8.75
26,Chicken Bowl,"[Roasted Chili Corn Salsa (Medium), [Pinto Bea...",8.49
42,Chicken Bowl,"[Roasted Chili Corn Salsa, [Rice, Black Beans,...",11.25


## 5. Sorting by Item Name and Price

In [5]:
# Alphabetical sorting of unique item names
unique_items_sorted = chipo["item_name"].drop_duplicates().sort_values()
print("First 5 menu items (A-Z):\n", unique_items_sorted.head())

First 5 menu items (A-Z):
 298         6 Pack Soft Drink
39              Barbacoa Bowl
21           Barbacoa Burrito
168     Barbacoa Crispy Tacos
1229      Barbacoa Salad Bowl
Name: item_name, dtype: str


In [6]:
# Most expensive single order line item in the dataset
most_expensive_order = chipo.sort_values(by="item_price", ascending=False).head(1)
print("Highest single line item order:")
display(most_expensive_order)

Highest single line item order:


,order_id,quantity,item_name,choice_description,item_price
3598,1443,15,Chips and Fresh Tomato Salsa,NaN,44.25


## 6. Frequency Queries: Orders vs Total Quantity

### ⚠️ Top Interview Question: Transactions vs Total Units
- Number of **orders/transactions** containing the item: `(chipo['item_name'] == 'Veggie Salad Bowl').sum()`.
- Total **units consumed**: `chipo.loc[chipo['item_name'] == 'Veggie Salad Bowl', 'quantity'].sum()`.

In [7]:
# Frequency of Veggie Salad Bowl
veggie_mask = chipo["item_name"] == "Veggie Salad Bowl"
times_ordered = veggie_mask.sum()
units_ordered = chipo.loc[veggie_mask, "quantity"].sum()

print(f"Veggie Salad Bowl was ordered in {times_ordered} separate line items.")
print(f"Total units of Veggie Salad Bowl sold: {units_ordered}")

Veggie Salad Bowl was ordered in 18 separate line items.
Total units of Veggie Salad Bowl sold: 18


In [8]:
# Orders with more than one Canned Soda
multi_soda_mask = (chipo["item_name"] == "Canned Soda") & (chipo["quantity"] > 1)
multi_soda_count = multi_soda_mask.sum()

print(f"Number of times someone ordered MORE than one Canned Soda: {multi_soda_count}")
chipo[multi_soda_mask].head(3)

Number of times someone ordered MORE than one Canned Soda: 20


,order_id,quantity,item_name,choice_description,item_price
18,9,2,Canned Soda,[Sprite],2.18
51,23,2,Canned Soda,[Mountain Dew],2.18
162,73,2,Canned Soda,[Diet Coke],2.18


## 7. Filtering & Order Analytics Cheat Sheet

| Task | Idiomatic Syntax | Performance / Gotcha |
| :--- | :--- | :--- |
| **Count Matching Rows** | `mask.sum()` | 3x faster than `len(df[mask])`; no memory allocation |
| **Isolate Base Prices** | `df[df['qty'] == 1].drop_duplicates('item')` | Eliminates multi-unit price distortion |
| **Multi-Condition** | `(cond1) & (cond2)` | Bitwise `&` with parentheses mandatory |
| **Text Search on Nulls** | `s.str.contains('text', na=False)` | `na=False` prevents `NaN` boolean errors |
| **Top-K Records** | `df.sort_values(col, ascending=False).head(K)` | Highly optimized in C |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: `mask.sum()` vs `len(df[mask])`
**Question**: When you only need to count how many rows satisfy a condition, why is `(df['A'] > 10).sum()` strictly superior to `len(df[df['A'] > 10])`?

**Answer**:
1. **Memory Allocation**: `df[df['A'] > 10]` allocates an entirely new DataFrame in memory containing all filtered rows and columns. For a 10M row table, this creates massive memory pressure.
2. **Speed**: `mask.sum()` sums the boolean vector in C/NumPy ($1$ for True, $0$ for False) without allocating a new DataFrame, running **3x to 10x faster**!

In [9]:
# Benchmark: mask.sum() vs len(df[mask])
import time

t0 = time.perf_counter()
for _ in range(500):
    c1 = (chipo["quantity"] > 1).sum()
t1 = time.perf_counter()

t2 = time.perf_counter()
for _ in range(500):
    c2 = len(chipo[chipo["quantity"] > 1])
t3 = time.perf_counter()

print(f"Time using mask.sum():      {(t1 - t0)*1000:.2f} ms")
print(f"Time using len(df[mask]):   {(t3 - t2)*1000:.2f} ms")
print(f"Speedup via mask.sum():     {((t3 - t2)/(t1 - t0)):.1f}x faster!")

Time using mask.sum():      9.22 ms
Time using len(df[mask]):   27.32 ms
Speedup via mask.sum():     3.0x faster!


### Q2: Handling `NaN` in `.str.contains()`
**Question**: Why does filtering on text columns like `df[df['choice_description'].str.contains('Salsa')]` sometimes raise `ValueError: Cannot mask with non-boolean array containing NA / NaN values`?

**Answer**:
If `choice_description` contains `NaN` values (missing toppings), `str.contains()` evaluates to `NaN` for those rows. A boolean filter mask cannot contain `NaN`!
**Solution**: Always pass `na=False` (or `na=True` if you want missing values included):
`df['choice_description'].str.contains('Salsa', na=False)`.

In [10]:
# Demonstration of na=False parameter
has_salsa = chipo["choice_description"].str.contains("Salsa", na=False)
print("Orders containing Salsa (safely ignoring NaNs):", has_salsa.sum())
chipo[has_salsa].head(3)

Orders containing Salsa (safely ignoring NaNs): 2808


,order_id,quantity,item_name,choice_description,item_price
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98
5,3,1,Chicken Bowl,"[Fresh Tomato Salsa (Mild), [Rice, Cheese, Sou...",10.98
7,4,1,Steak Burrito,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",11.75


### Q3: Advanced Interview Challenge: Most Popular Burrito Toppings
**Challenge**: Across all orders of Burritos (any item with "Burrito" in its name), extract the single most frequently selected salsa or ingredient from `choice_description`!

In [11]:
# Solution: filtering Burritos and exploding choice ingredients
burritos = chipo[chipo["item_name"].str.contains("Burrito", na=False)].copy()

# Clean brackets and split toppings into individual items
top_burrito_toppings = (
    burritos["choice_description"]
    .dropna()
    .str.replace(r"[\[\]]", "", regex=True)
    .str.split(", ")
    .explode()
    .str.strip()
    .value_counts()
    .head(5)
)

print("Top 5 most frequent toppings in Burritos:")
display(top_burrito_toppings)

Top 5 most frequent toppings in Burritos:


choice_description
Rice           1063
Cheese          960
Sour Cream      745
Lettuce         691
Black Beans     543
Name: count, dtype: int64